## هل يرتبط الترويج بزيادة الطلب؟ اختبار إحصائي

نقارن الطلب مع وجود مؤشر البريد الإلكتروني ومؤشر الظهور في الصفحة الرئيسية. النموذج الأساسي هو انحدار خطي على `num_orders` مع ضبط تأثيرات ثابتة لزوج المركز والوجبة، والمركز والأسبوع، والوجبة والأسبوع. ندخل المؤشرين معًا، ونحسب أخطاء معيارية متينة مجمعة حسب المركز. تعرض خلية النتائج فروق المتوسط الخام ثم الفروق المعدّلة وفواصل الثقة واختبار الفرضية الصفرية. كما تعرض نموذج حساسية يضبط سعري `checkout_price` و`base_price`؛ وقد يكون سعر الدفع جزءًا من آلية الترويج، لذا لا نعدّه تقديرًا للأثر الكلي.

In [ ]:
import math
import numpy as np

outcome = orders["num_orders"].to_numpy(dtype=float)
promotion_columns = ["emailer_for_promotion", "homepage_featured"]
promotion_flags = orders[promotion_columns].to_numpy(dtype=float)
fixed_effect_groups = [
    pd.MultiIndex.from_frame(orders[["center_id", "meal_id"]]).factorize(sort=True)[0],
    pd.MultiIndex.from_frame(orders[["center_id", "week"]]).factorize(sort=True)[0],
    pd.MultiIndex.from_frame(orders[["meal_id", "week"]]).factorize(sort=True)[0],
]
fixed_effect_levels = [int(group.max()) + 1 for group in fixed_effect_groups]
center_clusters = pd.factorize(orders["center_id"], sort=True)[0]
cluster_count = int(center_clusters.max()) + 1

def absorb_fixed_effects(values):
    residuals = values.astype(float, copy=True)
    for iteration in range(1000):
        for group, level_count in zip(fixed_effect_groups, fixed_effect_levels):
            counts = np.bincount(group, minlength=level_count).astype(float)
            for column in range(residuals.shape[1]):
                means = np.bincount(
                    group, weights=residuals[:, column], minlength=level_count
                ) / counts
                residuals[:, column] -= means[group]
        largest_group_mean = max(
            np.max(np.abs(
                np.bincount(group, weights=residuals[:, column], minlength=level_count)
                / np.bincount(group, minlength=level_count)
            ))
            for group, level_count in zip(fixed_effect_groups, fixed_effect_levels)
            for column in range(residuals.shape[1])
        )
        if largest_group_mean < 1e-8:
            return residuals
    raise RuntimeError("لم تتقارب إزالة التأثيرات الثابتة")

def beta_continued_fraction(a, b, x):
    qab, qap, qam = a + b, a + 1, a - 1
    c = 1.0
    denominator = 1 - qab * x / qap
    denominator = 1e-300 if abs(denominator) < 1e-300 else denominator
    denominator = 1 / denominator
    result = denominator
    for step in range(1, 301):
        twice = 2 * step
        term = step * (b - step) * x / ((qam + twice) * (a + twice))
        denominator = 1 + term * denominator
        denominator = 1e-300 if abs(denominator) < 1e-300 else denominator
        c = 1 + term / c
        c = 1e-300 if abs(c) < 1e-300 else c
        denominator = 1 / denominator
        result *= denominator * c
        term = -(a + step) * (qab + step) * x / ((a + twice) * (qap + twice))
        denominator = 1 + term * denominator
        denominator = 1e-300 if abs(denominator) < 1e-300 else denominator
        c = 1 + term / c
        c = 1e-300 if abs(c) < 1e-300 else c
        denominator = 1 / denominator
        delta = denominator * c
        result *= delta
        if abs(delta - 1) < 3e-14:
            break
    return result

def regularized_beta(a, b, x):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    factor = math.exp(
        math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
        + a * math.log(x) + b * math.log1p(-x)
    )
    if x < (a + 1) / (a + b + 2):
        return factor * beta_continued_fraction(a, b, x) / a
    return 1 - factor * beta_continued_fraction(a, b, 1 - x) / b

def student_t_two_sided_p_value(t_value, degrees_freedom):
    x = degrees_freedom / (degrees_freedom + t_value**2)
    return regularized_beta(degrees_freedom / 2, 0.5, x)

def fit_promotion_model(include_prices=False):
    regressors = promotion_flags
    if include_prices:
        prices = orders[["checkout_price", "base_price"]].to_numpy(dtype=float)
        regressors = np.column_stack([regressors, prices])
    residualized = absorb_fixed_effects(np.column_stack([outcome, regressors]))
    y_residual = residualized[:, 0]
    x_residual = residualized[:, 1:]
    gram = x_residual.T @ x_residual
    coefficients = np.linalg.solve(gram, x_residual.T @ y_residual)
    residual = y_residual - x_residual @ coefficients
    bread = np.linalg.inv(gram)
    meat = np.zeros_like(gram)
    for cluster in range(cluster_count):
        mask = center_clusters == cluster
        score = x_residual[mask].T @ residual[mask]
        meat += np.outer(score, score)
    parameter_count = sum(fixed_effect_levels) - (len(fixed_effect_levels) - 1) + len(coefficients)
    correction = (cluster_count / (cluster_count - 1)) * (
        (len(orders) - 1) / (len(orders) - parameter_count)
    )
    covariance = bread @ meat @ bread * correction
    standard_errors = np.sqrt(np.diag(covariance))
    return coefficients, standard_errors

raw_means = orders.groupby(promotion_columns)["num_orders"].agg(["count", "mean"])
print("المتوسط الخام حسب حالة المؤشرين (emailer, homepage):")
for flags, row in raw_means.iterrows():
    print(f"{flags}: عدد={int(row['count']):,}، متوسط الطلب={row['mean']:.2f}")

critical_value = 1.992  # t(0.975, 76), 77 مراكز
for adjust_prices in [False, True]:
    coefficients, standard_errors = fit_promotion_model(include_prices=adjust_prices)
    baseline = outcome.mean() - np.dot(coefficients[:2], promotion_flags.mean(axis=0))
    print("\nالنموذج: تأثيرات ثابتة + أسعار" if adjust_prices else "\nالنموذج الأساسي: تأثيرات ثابتة، دون ضبط الأسعار")
    print(f"خط أساس مقدر عند غياب المؤشرين: {baseline:.2f} طلبًا لكل سجل")
    for index, label in enumerate(["الترويج عبر البريد", "الظهور في الصفحة الرئيسية"]):
        t_value = coefficients[index] / standard_errors[index]
        p_value = student_t_two_sided_p_value(abs(t_value), cluster_count - 1)
        lower = coefficients[index] - critical_value * standard_errors[index]
        upper = coefficients[index] + critical_value * standard_errors[index]
        percent = coefficients[index] / baseline * 100
        print(f"{label}: فرق={coefficients[index]:.2f} طلبًا، نسبة={percent:.2f}%، "
              f"95% CI=[{lower:.2f}, {upper:.2f}]، t={t_value:.2f}، p={p_value:.3g}")

# التحليل الاستكشافي للطلب الأسبوعي على الوجبات

يقرأ هذا الدفتر ملفات `data/row/` دون تعديلها، ويربط بيانات التدريب بجدولي الوجبات ومراكز التوصيل مع التحقق من ثبات عدد الصفوف ومجموع الطلبات. كما يقارن مجموع الطلبات بين الأسبوعين 144 و145، ويستخدم متوسط الأسابيع 142–145 كمؤشر خط أساس تقديري للأسبوع 146، وليس كتنبؤ إحصائي مدرّب.

الأرقام الواردة في التقرير مستمدة من مخرجات أقسام هذا الدفتر.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd() / "data" / "row"
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
meals = pd.read_csv(DATA_DIR / "meal_info.csv")
centers = pd.read_csv(DATA_DIR / "fulfilment_center_info.csv")

required = {"week", "center_id", "meal_id", "num_orders"}
missing_columns = required - set(train.columns)
if missing_columns:
    raise ValueError(f"أعمدة مطلوبة غير موجودة: {sorted(missing_columns)}")

if meals["meal_id"].duplicated().any() or centers["center_id"].duplicated().any():
    raise ValueError("يجب أن تكون مفاتيح الجداول المرجعية فريدة")

before_rows = len(train)
before_orders = int(train["num_orders"].sum())
unmatched_meals = train.loc[~train["meal_id"].isin(meals["meal_id"]), "meal_id"].nunique()
orders = train.merge(meals, on="meal_id", how="left", validate="many_to_one")
after_meal_rows = len(orders)
after_meal_orders = int(orders["num_orders"].sum())
unmatched_centers = orders.loc[~orders["center_id"].isin(centers["center_id"]), "center_id"].nunique()
orders = orders.merge(centers, on="center_id", how="left", validate="many_to_one")
after_center_rows = len(orders)
after_center_orders = int(orders["num_orders"].sum())
print(f"صفوف التدريب: {len(train)}")
print(f"صفوف الاختبار: {len(test)}")
print(f"نطاق أسابيع الاختبار: {test['week'].min()}–{test['week'].max()}")
print(f"عمود num_orders موجود في الاختبار: {'num_orders' in test.columns}")
print(f"صفوف ملف الوجبات: {len(meals)}")
print(f"صفوف ملف المراكز: {len(centers)}")
print(f"صفوف الطلب قبل الربط: {before_rows}")
print(f"مجموع الطلبات قبل الربط: {before_orders}")
print(f"بعد ربط الوجبات | الصفوف: {after_meal_rows} | الطلبات: {after_meal_orders} | مفاتيح غير مطابقة: {unmatched_meals}")
print(f"بعد ربط المراكز | الصفوف: {after_center_rows} | الطلبات: {after_center_orders} | مفاتيح غير مطابقة: {unmatched_centers}")
if (after_meal_rows != before_rows or after_center_rows != before_rows
        or after_meal_orders != before_orders or after_center_orders != before_orders
        or unmatched_meals != 0 or unmatched_centers != 0):
    raise ValueError("فشل تحقق عدد الصفوف أو مجموع الطلبات أو مطابقة المفاتيح بعد الربط")
if train.duplicated(["week", "center_id", "meal_id"]).any():
    raise ValueError("توجد مفاتيح أسبوع/مركز/وجبة مكررة")

OUTPUT_DIR = Path.cwd() / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "train_enriched.csv"
orders.to_csv(OUTPUT_FILE, index=False)
print(f"تم حفظ البيانات المرتبطة في: {OUTPUT_FILE.relative_to(Path.cwd()).as_posix()}")

صفوف التدريب: 456548
صفوف الاختبار: 32573
نطاق أسابيع الاختبار: 146–155
عمود num_orders موجود في الاختبار: False
صفوف ملف الوجبات: 51
صفوف ملف المراكز: 77
صفوف الطلب قبل الربط: 456548
مجموع الطلبات قبل الربط: 119557485
بعد ربط الوجبات | الصفوف: 456548 | الطلبات: 119557485 | مفاتيح غير مطابقة: 0
بعد ربط المراكز | الصفوف: 456548 | الطلبات: 119557485 | مفاتيح غير مطابقة: 0
تم حفظ البيانات المرتبطة في: data/processed/train_enriched.csv


## إجمالي الطلب الأسبوعي

يجمع هذا القسم `num_orders` لجميع الوجبات والمراكز في كل أسبوع، ويقارن الأسبوع 145 بالأسبوع السابق 144.

In [2]:
weekly = (orders.groupby("week", as_index=False)
          .agg(total_orders=("num_orders", "sum"), rows=("num_orders", "size"))
          .sort_values("week"))
recent = weekly[weekly["week"].between(137, 145)]
print("week | total_orders | rows")
for row in recent.itertuples(index=False):
    print(f"{row.week} | {row.total_orders} | {row.rows}")

week_totals = weekly.set_index("week")["total_orders"]
wow_change = int(week_totals.loc[145] - week_totals.loc[144])
wow_pct = (week_totals.loc[145] / week_totals.loc[144] - 1) * 100
print(f"التغير من الأسبوع 144 إلى 145: {wow_change} طلبًا ({wow_pct:.2f}%)")

week | total_orders | rows
137 | 719740 | 3283
138 | 667343 | 3278
139 | 751935 | 3279
140 | 802689 | 3332
141 | 776240 | 3263
142 | 739661 | 3238
143 | 759926 | 3305
144 | 801197 | 3302
145 | 671926 | 3268
التغير من الأسبوع 144 إلى 145: -129271 طلبًا (-16.13%)


## مؤشرات الطلب حسب الوجبة

المؤشر التقديري لكل وجبة هو متوسط مجموع الطلبات الأسبوعي عبر الأسابيع 142–145، مع اعتبار غياب سجل وجبة/مركز في أسبوع على أنه لا يوجد طلب مسجل لذلك الزوج في ذلك الأسبوع. كما يقيس عمود التغير الفرق بين مجموع الطلبات في الأسبوعين 144 و145. هذا خط أساس بسيط لا نموذج تنبؤ.

In [3]:
meal_week = (orders[orders["week"].between(142, 145)]
             .groupby(["meal_id", "week"], as_index=False)
             .agg(orders=("num_orders", "sum")))
meal_pivot = (meal_week.pivot(index="meal_id", columns="week", values="orders")
              .reindex(columns=range(142, 146)).fillna(0))
meal_metrics = pd.DataFrame(index=meal_pivot.index)
meal_metrics["avg_4w"] = meal_pivot.mean(axis=1)
meal_metrics["week144"] = meal_pivot[144]
meal_metrics["week145"] = meal_pivot[145]
meal_metrics["change"] = meal_metrics["week145"] - meal_metrics["week144"]
meal_metrics["change_pct"] = meal_metrics["change"] / meal_metrics["week144"] * 100
meal_metrics = meal_metrics.reset_index().merge(
    meals, on="meal_id", how="left", validate="one_to_one")
print("أعلى خمس وجبات حسب متوسط الطلب الأسبوعي للأسابيع 142–145:")
print("meal_id | category | cuisine | avg_4w | week144 | week145 | change | change_pct")
for row in meal_metrics.nlargest(5, "avg_4w").itertuples(index=False):
    print(f"{row.meal_id} | {row.category} | {row.cuisine} | {row.avg_4w:.2f} | "
          f"{row.week144:.0f} | {row.week145:.0f} | {row.change:.0f} | {row.change_pct:.2f}%")

category_week = (orders[orders["week"].isin([144, 145])]
                 .groupby(["category", "week"])["num_orders"].sum().unstack(fill_value=0))
category_week["change"] = category_week[145] - category_week[144]
category_week["change_pct"] = category_week["change"] / category_week[144] * 100
print("\nتغير إجمالي الفئات، الأسبوع 144 إلى 145:")
for category in ["Beverages", "Rice Bowl", "Salad", "Seafood", "Desert"]:
    row = category_week.loc[category]
    print(f"{category} | {row[144]:.0f} | {row[145]:.0f} | "
          f"{row['change']:.0f} | {row['change_pct']:.2f}%")

week_rows = {week: part.set_index(["center_id", "meal_id"])["num_orders"]
             for week, part in orders[orders["week"].isin([144, 145])].groupby("week")}
common_pairs = week_rows[144].index.intersection(week_rows[145].index)
common_144 = int(week_rows[144].loc[common_pairs].sum())
common_145 = int(week_rows[145].loc[common_pairs].sum())
common_pct = (common_145 / common_144 - 1) * 100
print(f"\nالأزواج نفسها من المركز والوجبة الموجودة في الأسبوعين: {len(common_pairs)}")
print(f"مجموع الطلب لهذه الأزواج في الأسبوع 144: {common_144}")
print(f"مجموع الطلب لهذه الأزواج في الأسبوع 145: {common_145}")
print(f"التغير في الأزواج المشتركة: {common_pct:.2f}%")

أعلى خمس وجبات حسب متوسط الطلب الأسبوعي للأسابيع 142–145:
meal_id | category | cuisine | avg_4w | week144 | week145 | change | change_pct
1885 | Beverages | Thai | 56369.75 | 56871 | 54905 | -1966 | -3.46%
2290 | Rice Bowl | Indian | 46112.00 | 52756 | 28468 | -24288 | -46.04%
1754 | Sandwich | Italian | 42094.00 | 45377 | 42629 | -2748 | -6.06%
1109 | Rice Bowl | Indian | 41836.25 | 43042 | 42568 | -474 | -1.10%
2707 | Beverages | Italian | 38847.00 | 40484 | 39833 | -651 | -1.61%

تغير إجمالي الفئات، الأسبوع 144 إلى 145:
Beverages | 265886 | 235091 | -30795 | -11.58%
Rice Bowl | 118935 | 93183 | -25752 | -21.65%
Salad | 112883 | 85582 | -27301 | -24.19%
Seafood | 31412 | 5503 | -25909 | -82.48%
Desert | 12996 | 14406 | 1410 | 10.85%

الأزواج نفسها من المركز والوجبة الموجودة في الأسبوعين: 3117
مجموع الطلب لهذه الأزواج في الأسبوع 144: 789763
مجموع الطلب لهذه الأزواج في الأسبوع 145: 666460
التغير في الأزواج المشتركة: -15.61%


## فحص جودة البيانات

يفحص هذا القسم القيم المفقودة والتكرار والسعر والطلبات الشاذة وتغطية الأسابيع لكل وجبة. تُعرّف الطلبات الشاذة بطريقتين باستخدام حدود IQR (الربيع الأول ناقص 1.5×IQR، والربيع الثالث زائد 1.5×IQR): على كامل البيانات، وداخل كل وجبة على حدة لتقليل أثر اختلاف مستويات الطلب بين الوجبات. هذه إشارات للمراجعة وليست مبررًا لحذف السجلات. تُقاس فجوات الأسابيع على نطاق الأسابيع الموجودة في ملف التدريب، ولا يعني غياب سجل وجبة/أسبوع أن الطلب كان صفرًا.

In [4]:
from itertools import product

print(f"عدد الصفوف: {len(orders):,}")
print(f"عدد الخلايا المفقودة: {int(orders.isna().sum().sum()):,}")
print(f"الصفوف المكررة بالكامل: {int(orders.duplicated().sum()):,}")
business_key = ["week", "center_id", "meal_id"]
print(f"تكرار المفتاح (week, center_id, meal_id): {int(orders.duplicated(business_key).sum()):,}")
print(f"تكرار id: {int(orders['id'].duplicated().sum()):,}")

higher_price = orders["checkout_price"] > orders["base_price"]
equal_price = orders["checkout_price"] == orders["base_price"]
lower_price = orders["checkout_price"] < orders["base_price"]
print(f"checkout_price > base_price: {int(higher_price.sum()):,} ({higher_price.mean() * 100:.2f}%)")
print(f"checkout_price = base_price: {int(equal_price.sum()):,}")
print(f"checkout_price < base_price: {int(lower_price.sum()):,}")

q1, q3 = orders["num_orders"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
global_outliers = (orders["num_orders"] < lower_bound) | (orders["num_orders"] > upper_bound)
meal_q1 = orders.groupby("meal_id")["num_orders"].transform(lambda values: values.quantile(0.25))
meal_q3 = orders.groupby("meal_id")["num_orders"].transform(lambda values: values.quantile(0.75))
meal_iqr = meal_q3 - meal_q1
meal_outliers = ((orders["num_orders"] < meal_q1 - 1.5 * meal_iqr)
                 | (orders["num_orders"] > meal_q3 + 1.5 * meal_iqr))
print(f"IQR العام: Q1={q1:.0f}، Q3={q3:.0f}، الحدود=[{lower_bound:.0f}، {upper_bound:.0f}]")
print(f"طلبات خارج حدود IQR العام: {int(global_outliers.sum()):,}، كلها فوق الحد الأعلى؛ أكبر قيمة={int(orders['num_orders'].max()):,}")
print(f"طلبات خارج IQR داخل كل وجبة: {int(meal_outliers.sum()):,} ({meal_outliers.mean() * 100:.2f}%)")

expected_weeks = set(range(int(orders["week"].min()), int(orders["week"].max()) + 1))
meal_ids = sorted(orders["meal_id"].unique())
observed_pairs = pd.MultiIndex.from_frame(orders[["meal_id", "week"]].drop_duplicates())
expected_pairs = pd.MultiIndex.from_tuples(
    product(meal_ids, sorted(expected_weeks)), names=["meal_id", "week"])
missing_pairs = expected_pairs.difference(observed_pairs)
meals_with_gaps = sorted(missing_pairs.get_level_values("meal_id").unique())
print(f"نطاق الأسابيع: {min(expected_weeks)}–{max(expected_weeks)} ({len(expected_weeks)} أسبوعًا)")
print(f"وجبات بفجوات أسبوعية: {len(meals_with_gaps)} من {len(meal_ids)}؛ أزواج وجبة-أسبوع غير موجودة: {len(missing_pairs)}")
for meal_id in meals_with_gaps:
    absent = sorted(missing_pairs[missing_pairs.get_level_values("meal_id") == meal_id]
                    .get_level_values("week"))
    if absent == list(range(1, 59)):
        description = "الأسابيع 1–58 (58 أسبوعًا)"
    else:
        description = "الأسابيع " + ", ".join(map(str, absent))
    print(f"الوجبة {meal_id}: {description}")

عدد الصفوف: 456,548
عدد الخلايا المفقودة: 0
الصفوف المكررة بالكامل: 0
تكرار المفتاح (week, center_id, meal_id): 0
تكرار id: 0
checkout_price > base_price: 116,101 (25.43%)
checkout_price = base_price: 110,760
checkout_price < base_price: 229,687
IQR العام: Q1=54، Q3=324، الحدود=[-351، 729]
طلبات خارج حدود IQR العام: 32,937، كلها فوق الحد الأعلى؛ أكبر قيمة=24,299
طلبات خارج IQR داخل كل وجبة: 27,914 (6.11%)
نطاق الأسابيع: 1–145 (145 أسبوعًا)
وجبات بفجوات أسبوعية: 6 من 51؛ أزواج وجبة-أسبوع غير موجودة: 222
الوجبة 1571: الأسابيع 1–58 (58 أسبوعًا)
الوجبة 2104: الأسابيع 1–58 (58 أسبوعًا)
الوجبة 2490: الأسابيع 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 21, 22
الوجبة 2569: الأسابيع 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17
الوجبة 2664: الأسابيع 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13
الوجبة 2956: الأسابيع 1–58 (58 أسبوعًا)


## حدود التحليل

- يحتوي `test.csv` على الأسابيع 146–155 لكنه لا يحتوي على `num_orders`؛ لذلك لا يمكن التحقق منه كطلب فعلي أو اعتباره توقعًا منتجًا من هذا التحليل.
- متوسط الأسابيع الأربعة مؤشر تمهيدي بسيط، ولا يأخذ الأسعار أو العروض أو الموسمية أو اختلاف توافر الوجبات بين المراكز في الحسبان.
- مقارنة الأسبوعين تستخدم أيضًا الأزواج المتاحة في كليهما لفصل أثر اختلاف التغطية، لكنها لا تفسر أسباب تغير الطلب.
- أسماء الفئات مطبوعة كما وردت في ملف المصدر؛ مثل `Desert`.